# Install Required Libraries

In [1]:
!pip -q install chromadb openai langchain tiktoken


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


# Load Environment Variables (API Key)

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

# Import Libraries

In [23]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import RetrievalQA

# Download Dataset from Dropbox

In [24]:
import requests

url = "https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip?dl=1"

response = requests.get(url)

with open("new_articles.zip", "wb") as file:
    file.write(response.content)

print("Download complete")

Download complete


# Extract Zip

In [26]:
import zipfile

with zipfile.ZipFile("new_articles.zip", "r") as zip_ref:
    zip_ref.extractall("new_articles")

print("Files extracted successfully")

Files extracted successfully


# Load Documents from Local Folder

In [29]:
loader = DirectoryLoader(
    r"D:\Vector_DataBase\new_articles",   # ya "./new_articles" agar same folder me ho
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "latin-1"}
)

documents = loader.load()

# Split Documents into Chunks

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

text = text_splitter.split_documents(documents)

print(text[:2])

[Document(metadata={'source': 'D:\\Vector_DataBase\\new_articles\\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt'}, page_content='Signaling that investments in the supply chain sector remain robust, Pando, a startup developing fulfillment management technologies, today announced that it raised $30 million in a Series B round, bringing its total raised to $45 million.\n\nIron Pillar and Uncorrelated Ventures led the round, with participation from existing investors Nexus Venture Partners, Chiratae Ventures and Next47. CEO and founder Nitin Jayakrishnan says that the new capital will be put toward expanding Pandoâ\x80\x99s global sales, marketing and delivery capabilities.\n\nâ\x80\x9cWe will not expand into new industries or adjacent product areas,â\x80\x9d he told TechCrunch in an email interview. â\x80\x9cGreat talent is the foundation of the business â\x80\x94 we will continue to augment our teams at all levels of the organization. Pando is also open to explorin

# Create Embeddings and Vector Database (ChromaDB)

In [32]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

persist_directory = "db"

embedding = OpenAIEmbeddings()

vectordb = Chroma.from_documents(
    documents=text,
    embedding=embedding,
    persist_directory=persist_directory
)

# Persist Vector Database to Disk

In [33]:
vectordb.persist()

# Reload Vector Database from Disk

In [34]:
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

# Create Retriever from Vector Database

In [48]:
retriever = vectordb.as_retriever()

In [56]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Test Retrieval

In [61]:
docs = retriever.invoke("How much funding did Microsoft raise in the article?")
for doc in docs:
    print(doc.page_content)
    print("----")

April 28, 2023

VC firms including Sequoia Capital, Andreessen Horowitz, Thrive and K2 Global are picking up new shares, according to documents seen by TechCrunch. A source tells us Founders Fund is also investing. Altogether the VCs have put in just over $300 million at a valuation of $27 billion to $29 billion. This is separate to a big investment from Microsoft announced earlier this year, a person familiar with the development told TechCrunch, which closed in January. The size of Microsoftâs investment is believed to be around $10 billion, a figure we confirmed with our source.

April 25, 2023

Called ChatGPT Business, OpenAI describes the forthcoming offering as âfor professionals who need more control over their data as well as enterprises seeking to manage their end users.â
----
April 28, 2023

VC firms including Sequoia Capital, Andreessen Horowitz, Thrive and K2 Global are picking up new shares, according to documents seen by TechCrunch. A source tells us Founders Fund i

# Create Retrieval QA Chain

In [62]:
from langchain_classic.chains import RetrievalQA
from langchain_openai import OpenAI

qa_chain = RetrievalQA.from_chain_type(
    llm=OpenAI(),
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# Print Answer + Sources

In [65]:
def process_llm_response(response):
    print("Answer:\n")
    print(response['result'])
    
    print("\nSources:\n")
    
    sources = set()   # duplicate remove
    for doc in response['source_documents']:
        sources.add(doc.metadata['source'])
    
    for source in sources:
        print(source)

# Query the System / Ask Question

In [66]:
query = "How much funding did Microsoft raise?"
response = qa_chain(query)

process_llm_response(response)

Answer:

 The size of Microsoft's investment is believed to be around $10 billion.

Sources:

D:\Vector_DataBase\new_articles\05-03-chatgpt-everything-you-need-to-know-about-the-ai-powered-chatbot.txt
